# 04 — The test: January 2021 to October 2026, one nowcast a month

**Reads:** `data/processed/features.csv`, `results/tables/cv_choice.json`, `cv_best_by_feature_set.csv`.

**Writes:** `data/processed/nowcasts.csv`; `results/tables/test_metrics.csv`, `test_rmse_by_year.csv`,
`test_biggest_poll_misses.csv`, `test_feature_sets_not_used_for_choice.csv`.

**Procedure.** The settings and input sets from notebook 03 stay fixed. For each test month the models are
refitted on every month whose first print was out before that month's nowcast day, then make one nowcast.
That is what a forecaster would have done in real time: use all the history available, with the recipe
fixed in advance.

**Methods compared**

| Method | What it is |
|---|---|
| Last month's print | persistence: this month = last month |
| Average of the last 12 prints | the best rule without a model before 2021 (notebook 02) |
| Reuters poll | median forecast of the economists Reuters surveys before the release |
| Linear regression | ordinary least squares on its chosen inputs: the simplest model |
| Random forest | many deep-ish trees, each on a resample of months, averaged |
| Gradient boosting (XGBoost) | small trees added one at a time, each fixing the last ones' errors |
| Average of the two | equal-weight average of the random forest and XGBoost nowcasts |

**Scores:** root mean squared error (RMSE, in percentage points) over all test months, over 2021 and over
2022–2026; the Diebold–Mariano test of equal squared error against the poll and against persistence
(Harvey–Leybourne–Newbold small-sample version).

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"
for d in (PROC, TAB, FIG):
    d.mkdir(parents=True, exist_ok=True)

from scipy import stats
from nowcast_setup import TEST_START, FEATURE_SETS, MODEL_LABELS, make_model, load_features

feat = load_features(PROC / "features.csv")
choice = json.loads((TAB / "cv_choice.json").read_text())
test = feat.loc[TEST_START:]
print(len(test), "test months:", test.index.min(), "to", test.index.max())
{k: (v["feature_set"], v["params"]) for k, v in choice.items()}

68 test months: 2021-01 to 2026-10


{'rf': ('core + jet fuel',
  {'max_depth': 2, 'min_samples_leaf': 5, 'max_features': 0.5}),
 'xgb': ('core + jet fuel',
  {'n_estimators': 300,
   'learning_rate': 0.01,
   'max_depth': 1,
   'min_child_weight': 5}),
 'ols': ('core + used cars', {})}

## One nowcast per test month, each model refitted on the history available that day

In [2]:
def nowcast_path(kind, cols, params):
    out = []
    for t in test.index:
        train = feat.loc[: t - 1]
        assert (train["release_date"] < test.at[t, "nowcast_date"]).all()
        model = make_model(kind, params).fit(train[cols], train["y"])
        out.append(float(model.predict(test.loc[[t], cols])[0]))
    return pd.Series(out, index=test.index)


now = test[["release_date", "y", "poll", "core_last", "core_avg12"]].rename(
    columns={"core_last": "last_print", "core_avg12": "avg12"})
for kind in ("ols", "rf", "xgb"):
    now[kind] = nowcast_path(kind, FEATURE_SETS[choice[kind]["feature_set"]], choice[kind]["params"])
now["average"] = (now["rf"] + now["xgb"]) / 2

out = now.copy()
out.index = out.index.astype(str)
out.to_csv(PROC / "nowcasts.csv", float_format="%.6f")
now.loc["2021-03":"2021-07"].drop(columns="release_date").round(2)

,y,poll,last_print,avg12,ols,rf,xgb,average
period,,,,,,,,
2021-03,0.3,0.2,0.1,0.10,0.23,0.16,0.16,0.16
2021-04,0.9,0.3,0.3,0.13,0.37,0.18,0.18,0.18
2021-05,0.7,0.4,0.9,0.24,0.62,0.18,0.19,0.18
2021-06,0.9,0.4,0.7,0.31,0.39,0.21,0.22,0.21
2021-07,0.3,0.4,0.9,0.37,0.49,0.28,0.33,0.30


## Scores

In [3]:
METHODS = {
    "last_print": "Last month's print (persistence)",
    "avg12": "Average of the last 12 prints",
    "poll": "Reuters poll (median)",
    "ols": "Linear regression",
    "rf": "Random forest",
    "xgb": "Gradient boosting (XGBoost)",
    "average": "Average of random forest and XGBoost",
}
Y2021 = now.index.year == 2021
LATER = now.index.year >= 2022


def rmse(e):
    return float(np.sqrt(np.mean(np.square(e))))


def dm_test(e1, e2):
    """Diebold-Mariano test of equal mean squared error, one-step-ahead, with the Harvey-Leybourne-Newbold
    small-sample correction. Negative statistic: the first method has the smaller squared errors."""
    d = np.square(np.asarray(e1)) - np.square(np.asarray(e2))
    n = len(d)
    stat = d.mean() / np.sqrt(d.var() / n) * np.sqrt((n - 1) / n)
    return float(stat), float(2 * stats.t.sf(abs(stat), df=n - 1))


err = {m: now["y"] - now[m] for m in METHODS}
rows = []
for m, label in METHODS.items():
    e = err[m]
    dm_poll = dm_test(e, err["poll"]) if m != "poll" else (np.nan, np.nan)
    dm_last = dm_test(e, err["last_print"]) if m != "last_print" else (np.nan, np.nan)
    rows.append({
        "method": label,
        "RMSE, all test months": rmse(e),
        "RMSE, 2021": rmse(e[Y2021]),
        "RMSE, 2022-2026": rmse(e[LATER]),
        "MAE, all test months": float(e.abs().mean()),
        "RMSE, nowcast rounded to 0.1": rmse(now["y"] - now[m].round(1)),
        "closer than the poll (share of months)": float((e.abs() < err["poll"].abs()).mean()) if m != "poll" else np.nan,
        "DM statistic vs poll": dm_poll[0], "DM p-value vs poll": dm_poll[1],
        "DM statistic vs persistence": dm_last[0], "DM p-value vs persistence": dm_last[1],
        "months": len(e),
    })
metrics = pd.DataFrame(rows).set_index("method")
metrics.round(4).to_csv(TAB / "test_metrics.csv")
metrics.iloc[:, :7].round(3)

,"RMSE, all test months","RMSE, 2021","RMSE, 2022-2026","MAE, all test months","RMSE, nowcast rounded to 0.1",closer than the poll (share of months),DM statistic vs poll
method,,,,,,,
Last month's print (persistence),0.176,0.301,0.134,0.124,0.176,0.221,1.279
Average of the last 12 prints,0.180,0.344,0.119,0.125,0.185,0.382,2.216
Reuters poll (median),0.149,0.268,0.108,0.101,0.149,NaN,NaN
Linear regression,0.147,0.238,0.119,0.106,0.145,0.426,-0.220
Random forest,0.189,0.368,0.120,0.128,0.186,0.368,2.498
Gradient boosting (XGBoost),0.186,0.361,0.119,0.124,0.186,0.353,2.396
Average of random forest and XGBoost,0.187,0.364,0.119,0.125,0.188,0.353,2.417


In [4]:
metrics.iloc[:, 7:].round(3)

,DM p-value vs poll,DM statistic vs persistence,DM p-value vs persistence,months
method,,,,
Last month's print (persistence),0.205,NaN,NaN,68
Average of the last 12 prints,0.030,0.183,0.855,68
Reuters poll (median),NaN,-1.279,0.205,68
Linear regression,0.826,-1.402,0.165,68
Random forest,0.015,0.493,0.624,68
Gradient boosting (XGBoost),0.019,0.399,0.691,68
Average of random forest and XGBoost,0.018,0.425,0.672,68


## By calendar year

In [5]:
by_year = pd.DataFrame({METHODS[m]: err[m].groupby(err[m].index.year).apply(rmse) for m in METHODS})
by_year["months"] = err["poll"].groupby(err["poll"].index.year).size()
by_year.round(4).to_csv(TAB / "test_rmse_by_year.csv")
by_year.round(3)

,Last month's print (persistence),Average of the last 12 prints,Reuters poll (median),Linear regression,Random forest,Gradient boosting (XGBoost),Average of random forest and XGBoost,months
period,,,,,,,,
2021,0.301,0.344,0.268,0.238,0.368,0.361,0.364,12
2022,0.206,0.180,0.158,0.199,0.182,0.182,0.181,12
2023,0.091,0.101,0.058,0.071,0.099,0.102,0.100,12
2024,0.076,0.083,0.076,0.063,0.084,0.082,0.082,12
2025,0.118,0.089,0.118,0.083,0.093,0.091,0.091,10
2026,0.138,0.107,0.100,0.118,0.109,0.104,0.106,10


## The poll's biggest misses in the test, and what each method said

In [6]:
miss = now.assign(poll_miss=now["y"] - now["poll"]).reindex(
    (now["y"] - now["poll"]).abs().sort_values(ascending=False).index[:8])
miss = miss[["y", "poll", "poll_miss", "last_print", "avg12", "ols", "rf", "xgb", "average"]]
out = miss.copy()
out.index = out.index.astype(str)
out.round(3).to_csv(TAB / "test_biggest_poll_misses.csv")
miss.round(2)

,y,poll,poll_miss,last_print,avg12,ols,rf,xgb,average
period,,,,,,,,,
2021-04,0.9,0.3,0.6,0.3,0.13,0.37,0.18,0.18,0.18
2021-06,0.9,0.4,0.5,0.7,0.31,0.39,0.21,0.22,0.21
2022-08,0.6,0.3,0.3,0.3,0.47,0.28,0.37,0.35,0.36
2021-05,0.7,0.4,0.3,0.9,0.24,0.62,0.18,0.19,0.18
2022-07,0.3,0.5,-0.2,0.7,0.47,0.53,0.53,0.52,0.52
2022-10,0.3,0.5,-0.2,0.6,0.54,0.50,0.55,0.50,0.52
2021-01,0.0,0.2,-0.2,0.1,0.12,0.15,0.17,0.17,0.17
2022-03,0.3,0.5,-0.2,0.5,0.52,0.33,0.46,0.50,0.48


## Not used for any choice: every input set on the test months

The input sets were chosen by cross-validation before 2021 (notebook 03). This table shows, after the fact,
how each input set would have done in the test with its own best settings. It is reported for
interpretation only and changes nothing above.

In [7]:
best = pd.read_csv(TAB / "cv_best_by_feature_set.csv")
rows = []
for _, r in best.iterrows():
    path = nowcast_path(r["model"], FEATURE_SETS[r["feature_set"]], json.loads(r["params"]))
    e = now["y"] - path
    rows.append({"model": MODEL_LABELS[r["model"]], "input set": r["feature_set"],
                 "chosen by CV": r["feature_set"] == choice[r["model"]]["feature_set"],
                 "CV RMSE (before 2021)": r["cv_rmse"], "test RMSE": rmse(e), "test RMSE, 2021": rmse(e[Y2021]),
                 "test RMSE, 2022-2026": rmse(e[LATER])})
supp = pd.DataFrame(rows)
supp.round(4).to_csv(TAB / "test_feature_sets_not_used_for_choice.csv", index=False)
supp.round(3)

,model,input set,chosen by CV,CV RMSE (before 2021),test RMSE,"test RMSE, 2021","test RMSE, 2022-2026"
0,Linear regression,core + used cars,True,0.101,0.147,0.238,0.119
1,Linear regression,core + jet fuel,False,0.102,0.166,0.306,0.116
2,Linear regression,core + used cars + jet fuel,False,0.102,0.147,0.245,0.116
3,Linear regression,core,False,0.102,0.168,0.312,0.116
4,Random forest,core + jet fuel,True,0.105,0.189,0.368,0.120
5,Random forest,core,False,0.106,0.189,0.370,0.118
6,Random forest,core + used cars + jet fuel,False,0.106,0.180,0.298,0.142
7,Random forest,core + used cars,False,0.106,0.209,0.373,0.152
8,Gradient boosting (XGBoost),core + jet fuel,True,0.104,0.186,0.361,0.119
9,Gradient boosting (XGBoost),core + used cars + jet fuel,False,0.104,0.178,0.300,0.138
